# Add a New GAN Style to the Gallery

Use this notebook for **GAN-based** models: CycleGAN (PyTorch) and AnimeGAN-style (TensorFlow/NHWC).
These models require an explicit tensor-layout selection — see the dropdown in cell 5.

Run cells **1 → 5** in order.  
Cell 3 opens a file-picker dialog; cell 4 shows model file info; cell 5 takes the style name, description, and layout; cell 6 writes everything to disk.

---

### Tensor layout options

| Option | Layout tag | Shape | Pixel range | Used by |
|--------|-----------|-------|-------------|--------|
| CycleGAN (PyTorch) | `nchw_tanh` | [1, 3, H, W] | [-1, 1] | CycleGAN style_monet, style_vangogh, style_cezanne, style_ukiyoe |
| AnimeGAN / TF NHWC | `nhwc_tanh` | [1, H, W, 3] | [-1, 1] | AnimeGANv3 (TensorFlow export) |

**For CNN (TransformerNet) models** use `add_CNN_style.ipynb` instead.

---

### Model file formats

| File | What it is | Needed at… |
|---|---|---|
| `.onnx` | Open Neural Network Exchange. Self-contained inference graph **+ embedded weights**. | Runtime (app uses this) |
| `.onnx.data` | External weight blob (>2 GB models). **Must stay in the same folder as `.onnx`.** | Runtime (alongside `.onnx`) |
| `.pth` | PyTorch state-dict — backup only, not used by the app. | Training / re-export only |

In [ ]:
import sys, pathlib

# -- make scripts/ importable
_scripts_dir = pathlib.Path(".").resolve()
if str(_scripts_dir) not in sys.path:
    sys.path.insert(0, str(_scripts_dir))

from add_style_helper import setup

ctx = setup()

In [ ]:
from add_style_helper import pick_onnx_file

paths = pick_onnx_file()

In [ ]:
from add_style_helper import report_model_files

report_model_files(paths.onnx_path, paths.pth_path, paths.data_path)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from add_style_helper import validate_style_id

name_w = widgets.Text(
    placeholder="e.g. Monet",
    description="Style name:",
    layout=widgets.Layout(width="380px"),
)
desc_w = widgets.Text(
    placeholder="e.g. Impressionist painting style — Claude Monet (CycleGAN)",
    description="Description:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "90px"},
)
author_w = widgets.Text(
    value="PeterWazinski",
    description="Author:",
    layout=widgets.Layout(width="380px"),
)
layout_w = widgets.Dropdown(
    options=[
        ("CycleGAN  (PyTorch NCHW tanh)", "nchw_tanh"),
        ("AnimeGANv3 / TF  (NHWC tanh)",  "nhwc_tanh"),
    ],
    value="nchw_tanh",
    description="Layout:",
    layout=widgets.Layout(width="520px"),
    style={"description_width": "90px"},
)
status_out = widgets.Output()

def _check(_: object = None) -> None:
    with status_out:
        status_out.clear_output(wait=True)
        raw = name_w.value.strip()
        if not raw:
            return
        _, msg = validate_style_id(raw, ctx.existing_ids)
        print(f"{msg}  tensor_layout='{layout_w.value}'")

name_w.observe(_check, names="value")
layout_w.observe(_check, names="value")
display(widgets.VBox([name_w, desc_w, author_w, layout_w, status_out]))

In [ ]:
from add_style_helper import install_style

style_id = install_style(
    onnx_path=paths.onnx_path,
    pth_path=paths.pth_path,
    data_path=paths.data_path,
    style_name=name_w.value,
    style_desc=desc_w.value,
    style_author=author_w.value,
    tensor_layout=layout_w.value,
    styles_dir=ctx.styles_dir,
    catalog_path=ctx.catalog_path,
    catalog=ctx.catalog,
    existing_ids=ctx.existing_ids,
    content_image=ctx.content_image,
    repo_root=ctx.repo_root,
)

from IPython.display import display, Image as IPyImage
preview_path = ctx.styles_dir / style_id / "preview.jpg"
if preview_path.exists():
    print(f"\nPreview thumbnail:")
    display(IPyImage(filename=str(preview_path), width=256))